# Exploratory Data Analysis

This notebook performs comprehensive exploratory data analysis on the dataset used for the **AdvancedClassifier** project. We examine data distributions, identify missing values, explore correlations between features, and visualize key relationships to inform feature engineering and model selection.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('Libraries loaded successfully.')

## 1. Load the Dataset

In [ ]:
train_df = pd.read_csv('../datasets/train.csv')
test_df = pd.read_csv('../datasets/test.csv')
val_df = pd.read_csv('../datasets/validation.csv')

print(f'Training set shape:   {train_df.shape}')
print(f'Validation set shape: {val_df.shape}')
print(f'Test set shape:       {test_df.shape}')

train_df.head()

## 2. Descriptive Statistics

We use `.describe()` to get summary statistics and `.info()` to inspect data types and non-null counts.

In [ ]:
train_df.describe().T

In [ ]:
train_df.info()

## 3. Missing Values Analysis

Detect and visualize missing values across all columns.

In [ ]:
missing = train_df.isnull().sum()
missing_pct = (missing / len(train_df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if missing_df.empty:
    print('No missing values found in the training set.')
else:
    print(missing_df)
    plt.figure(figsize=(10, 5))
    sns.barplot(x=missing_df.index, y='Missing %', data=missing_df, palette='Reds_r')
    plt.title('Percentage of Missing Values by Column')
    plt.xlabel('Column')
    plt.ylabel('Missing (%)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 4. Correlation Heatmap

A Pearson correlation matrix helps identify multicollinearity and the most influential features.

In [ ]:
numeric_cols = train_df.select_dtypes(include=[np.number]).columns
corr_matrix = train_df[numeric_cols].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8}
)
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Feature Distributions (Histograms)

Examine the distribution of each numeric feature to detect skewness and potential outliers.

In [ ]:
n_features = len(numeric_cols)
n_cols = 4
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(train_df[col], kde=True, ax=axes[i], color='steelblue', edgecolor='white')
    axes[i].set_title(f'{col}', fontsize=11)
    axes[i].set_xlabel('')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Feature Distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 6. Scatter Plots (Feature vs. Target)

Visualize pairwise relationships between selected features and the target variable.

In [ ]:
target_col = 'target'
features_to_plot = numeric_cols.drop(target_col, errors='ignore')[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(features_to_plot):
    sns.scatterplot(x=feat, y=target_col, data=train_df, ax=axes[i], alpha=0.4, edgecolor='none')
    axes[i].set_title(f'{feat} vs {target_col}', fontsize=11)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Scatter Plots: Selected Features vs Target', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 7. Key Takeaways

- Review descriptive statistics for unusual ranges or scales.
- Identify columns with significant missing values for imputation strategies.
- Note highly correlated feature pairs (|r| > 0.8) to consider feature pruning.
- Check feature distributions for heavy skewness that may benefit from log or Box-Cox transforms.
- Use scatter plots to spot non-linear relationships that inform model architecture decisions.